In [4]:
from sklearn.metrics import mean_squared_error
import random
import numpy as np
import pandas as pd


# Load and pre-process data

In [8]:
target_col = "log_average_ms"

node_cols = ["Abs", "Acos", "Add", "ai.onnx.ml::CategoryMapper", "And", "ArgMax", 
             "AveragePool", "BatchNormalization", "Cast", "Ceil", "Clip", 
             "com.microsoft::BiasGelu", "com.microsoft::DynamicQuantizeLSTM", 
             "com.microsoft::DynamicQuantizeMatMul", "com.microsoft::FastGelu", 
             "com.microsoft::FusedConv", "com.microsoft::FusedGemm", 
             "com.microsoft::FusedMatMul", "com.microsoft::MatMulIntegerToFloat", 
             "com.microsoft::QGemm", "com.microsoft::QLinearAdd", 
             "com.microsoft::QLinearAveragePool", "com.microsoft::QLinearConcat", 
             "com.microsoft::QLinearGlobalAveragePool", "com.microsoft::QLinearLeakyRelu", 
             "com.microsoft::QLinearMul", "com.microsoft::QLinearSigmoid", 
             "com.microsoft::QuickGelu", "com.microsoft::SkipLayerNormalization", 
             "Compress", "Concat", "Constant", "ConstantOfShape", "Conv", "ConvInteger", 
             "ConvTranspose", "Cos", "CumSum", "DequantizeLinear", "Div", "Dropout", 
             "DynamicQuantizeLinear", "Einsum", "Equal", "Erf", "Exp", "Expand", 
             "EyeLike", "Flatten", "Floor", "Gather", "GatherElements", "GatherND", 
             "Gelu", "Gemm", "GlobalAveragePool", "Greater", "GreaterOrEqual", "Hardmax", 
             "HardSigmoid", "HardSwish", "Identity", "If", "InstanceNormalization", 
             "LayerNormalization", "LeakyRelu", "Less", "LessOrEqual", "local::preprocess", 
             "Log", "LogSoftmax", "Loop", "LRN", "LSTM", "MatMul", "MatMulInteger", 
             "Max", "MaxPool", "Min", "Mod", "Mul", "Neg", "NonMaxSuppression", "NonZero", 
             "Not", "OneHot", "Or", "Pad", "Pow", "PRelu", "QLinearConv", "QLinearMatMul", 
             "QuantizeLinear", "Range", "Reciprocal", "ReduceMax", "ReduceMean", "ReduceMin", 
             "ReduceProd", "ReduceSum", "Relu", "Reshape", "Resize", "RoiAlign", "Round", 
             "Scan", "ScatterElements", "ScatterND", "Shape", "Sigmoid", 
             "SimplifiedLayerNormalization", "Sin", "Slice", "Softmax", "Split", "Sqrt", 
             "Squeeze", "Sub", "Sum", "Tanh", "Tile", "TopK", "Transpose", "Trilu", 
             "Unsqueeze", "Where", "Xor"]

model_feature_cols = node_cols + ["conv_flops", "matmul_flops",
                "elementwise_mb", "reduction_mb", "normalization_mb",
                "movement_mb", "memory_mb"]

hardware_feature_cols = ["l1d_cache_kb", "l1i_cache_kb", "l2_cache_kb", 
                         "base_clock_mhz", "num_cores", "memory_bandwith_gbs", 
                         "cpu_provider"]

metadata_cols = ["model", "cpu_provider",
                 "machine_type", "platform", "run_id"]

In [2]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [9]:
# load test, train, val sets

train_df = pd.read_csv("/content/drive/MyDrive/Data/train_set.csv")
val_df = pd.read_csv("/content/drive/MyDrive/Data/val_set.csv")
test_df = pd.read_csv("/content/drive/MyDrive/Data/test_set.csv")

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (100252, 154)
val: (21483, 154)
test: (21483, 154)


In [10]:
assert list(train_df.columns) == list(val_df.columns) == list(test_df.columns)

train_df.head()

,model,input_dimensions,input_dtypes,output_dimensions,output_dtypes,conv_flops,matmul_flops,elementwise_mb,reduction_mb,normalization_mb,...,base_clock_mhz,memory_bandwith_gbs,cpu_provider,machine_type,platform,repo_file,average_ms,stddev_ms,min_ms,max_ms
0,hardcorenas_d_Opset17_extended.onnx,x:1x3x224x224,x:float32,668:1x1000,668:float32,472440512,2560000,50.528229,5.204269,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,hardcorenas_d_Opset17.onnx,14.069188,0.082284,13.931521,14.247209
1,vit_base_patch8_224_in21k_Opset17_disable_all....,x:1x3x224x224,x:float32,1089:1x21843,1089:float32,231211008,156097479168,1771.169711,0.000000,792.141724,...,2450.000,205,amd,epyc,gcloud,vit_base_patch8_224_in21k_Opset17.onnx,1481.456045,150.199862,1094.324125,1632.506909
2,tf_efficientnetv2_m_in21ft1k_Opset17_basic.onnx,x:1x3x384x384,x:float32,2466:1x1000,2466:float32,31470991360,2560000,1201.339462,67.736023,0.000000,...,2449.998,205,amd,epyc,gcloud,tf_efficientnetv2_m_in21ft1k_Opset17.onnx,301.127424,12.889699,286.464729,324.336829
3,shufflenet_v2_x1_5_Opset16.onnx,x:1x3x224x224,x:float32,1137:1x1000,1137:float32,266034528,2048000,8.246674,1.630875,0.000000,...,2449.998,205,amd,epyc,gcloud,shufflenet_v2_x1_5_Opset16.onnx,8.161246,0.036559,8.080030,8.197800
4,wide_resnet101_2_Opset16_timm.onnx,x:1x3x224x224,x:float32,971:1x1000,971:float32,45502005248,4096000,252.656250,4.218750,0.000000,...,2300.000,717,intel,xeon_plat,bluehive,wide_resnet101_2_Opset16_timm.onnx,38.147223,0.068437,38.035275,38.292409


In [11]:
import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler

SEED = 67
random.seed(SEED)
np.random.seed(SEED)

In [12]:
# pre-processing function

def preprocess(df: pd.DataFrame, train=False) -> pd.DataFrame:

  #drop rows with missing target
  df = df.dropna(subset=["average_ms"])

  # keep Run 6/10's strict CV rule on all splits
  cv_limit = 0.1
  df["cv"] = df["stddev_ms"] / df["average_ms"]
  df = df[df["cv"] <= cv_limit].copy()

  #add log_average_ms column
  df["log_average_ms"] = np.log(df["average_ms"])

  #remove decimals and round down
  for col in ["elementwise_mb", "reduction_mb", "normalization_mb", "movement_mb"]:
    df[col] = df[col].round(0)

  return df

In [13]:
train_df = preprocess(train_df, True)
test_df = preprocess(test_df, False)
val_df = preprocess(val_df, False)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)

train: (95814, 156)
val: (20494, 156)
test: (20528, 156)


In [61]:
for col in model_feature_cols:
    train_df[col] = np.log1p(train_df[col].clip(lower=0))
    val_df[col] = np.log1p(val_df[col].clip(lower=0))
    test_df[col] = np.log1p(test_df[col].clip(lower=0))

# Data Setup

In [62]:
target_col = "log_average_ms"

In [63]:
cpu_provider_map = {
    "intel": 0.0,
    "amd": 1.0,
}

for df in [train_df, val_df, test_df]:
    df["cpu_provider_binary"] = df["cpu_provider"].map(cpu_provider_map)

    if df["cpu_provider_binary"].isna().any():
        bad_values = df.loc[df["cpu_provider_binary"].isna(), "cpu_provider"].unique()
        raise ValueError(f"Unknown cpu_provider values: {bad_values}")

hardware_feature_cols = [
    col for col in hardware_feature_cols
    if col not in {"cpu_provider", "cpu_provider_binary"}
]
hardware_feature_cols.append("cpu_provider_binary")

In [17]:
X_model_train = train_df[model_feature_cols].astype("float32").to_numpy()
X_model_val = val_df[model_feature_cols].astype("float32").to_numpy()
X_model_test = test_df[model_feature_cols].astype("float32").to_numpy()

X_hw_train = train_df[hardware_feature_cols].astype("float32").to_numpy()
X_hw_val = val_df[hardware_feature_cols].astype("float32").to_numpy()
X_hw_test = test_df[hardware_feature_cols].astype("float32").to_numpy()

y_train = train_df[target_col].astype("float32").to_numpy()
y_val = val_df[target_col].astype("float32").to_numpy()
y_test = test_df[target_col].astype("float32").to_numpy()

In [18]:
model_scaler = StandardScaler()
hw_scaler = StandardScaler()

X_model_train = model_scaler.fit_transform(X_model_train).astype("float32")
X_model_val = model_scaler.transform(X_model_val).astype("float32")
X_model_test = model_scaler.transform(X_model_test).astype("float32")

X_hw_train = hw_scaler.fit_transform(X_hw_train).astype("float32")
X_hw_val = hw_scaler.transform(X_hw_val).astype("float32")
X_hw_test = hw_scaler.transform(X_hw_test).astype("float32")

# Model Setup

In [19]:
def rmse_log(y_true, y_pred):
  return np.sqrt(mean_squared_error(y_true, y_pred))

In [20]:
import torch
from torch.utils.data import Dataset, DataLoader

In [21]:
class TwoTowerLatencyDataset(Dataset):
    def __init__(self, X_model, X_hw, y):
        self.X_model = torch.from_numpy(X_model)
        self.X_hw = torch.from_numpy(X_hw)
        self.y = torch.from_numpy(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_model[idx], self.X_hw[idx], self.y[idx]

In [22]:
batch_size = 1024

train_loader = DataLoader(
    TwoTowerLatencyDataset(X_model_train, X_hw_train, y_train),
    batch_size=batch_size,
    shuffle=True,
    num_workers=0,
)

val_loader = DataLoader(
    TwoTowerLatencyDataset(X_model_val, X_hw_val, y_val),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

test_loader = DataLoader(
    TwoTowerLatencyDataset(X_model_test, X_hw_test, y_test),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

In [23]:
import torch.nn as nn
import torch.nn.functional as F

In [24]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, dropout=0.10):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, dim * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(dim * 2, dim),
        )
        self.norm = nn.LayerNorm(dim)

    def forward(self, x):
        return self.norm(x + self.net(x))


class GatedFiLMLatencyMLP(nn.Module):
    def __init__(self, n_model_features: int, n_hw_features: int):
        super().__init__()

        emb_dim = 192

        self.model_in = nn.Sequential(
            nn.Linear(n_model_features, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.12),
            nn.Linear(384, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU(),
        )

        self.model_blocks = nn.Sequential(
            ResidualBlock(emb_dim, dropout=0.10),
            ResidualBlock(emb_dim, dropout=0.08),
            ResidualBlock(emb_dim, dropout=0.05),
        )

        self.hw_tower = nn.Sequential(
            nn.Linear(n_hw_features, 96),
            nn.LayerNorm(96),
            nn.GELU(),
            nn.Dropout(0.05),
            nn.Linear(96, emb_dim),
            nn.LayerNorm(emb_dim),
            nn.GELU(),
        )

        self.hw_film = nn.Linear(emb_dim, emb_dim * 2)
        self.gate = nn.Sequential(
            nn.Linear(emb_dim * 2, emb_dim),
            nn.Sigmoid(),
        )

        self.fusion = nn.Sequential(
            nn.Linear(emb_dim * 5, 384),
            nn.LayerNorm(384),
            nn.GELU(),
            nn.Dropout(0.12),

            ResidualBlock(384, dropout=0.08),

            nn.Linear(384, 192),
            nn.LayerNorm(192),
            nn.GELU(),
            nn.Dropout(0.05),

            nn.Linear(192, 1),
        )

    def forward(self, model_x, hw_x):
        model_emb = self.model_in(model_x)
        model_emb = self.model_blocks(model_emb)

        hw_emb = self.hw_tower(hw_x)

        gamma, beta = self.hw_film(hw_emb).chunk(2, dim=1)
        film_model = model_emb * (1 + gamma) + beta

        gate = self.gate(torch.cat([model_emb, hw_emb], dim=1))
        conditioned_model = gate * film_model + (1 - gate) * model_emb

        interaction = conditioned_model * hw_emb
        diff = torch.abs(conditioned_model - hw_emb)
        ratio_proxy = conditioned_model / (torch.abs(hw_emb) + 1.0)

        fused = torch.cat(
            [conditioned_model, hw_emb, interaction, diff, ratio_proxy],
            dim=1,
        )

        return self.fusion(fused).squeeze(1)

In [6]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# random.seed(SEED)
# np.random.seed(SEED)
# torch.manual_seed(SEED)
# if torch.cuda.is_available():
#     torch.cuda.manual_seed_all(SEED)

print("device:", device)

device: cuda


In [25]:
print("model features:", X_model_train.shape[1])
print("hardware features:", X_hw_train.shape[1])

model features: 134
hardware features: 7


In [26]:
model = GatedFiLMLatencyMLP(
    n_model_features=X_model_train.shape[1],
    n_hw_features=X_hw_train.shape[1],
).to(device)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=8e-4,
    weight_decay=2e-4,
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
    optimizer,
    T_0=250,
    T_mult=2,
    eta_min=1e-5,
)

loss_fn = nn.SmoothL1Loss(beta=0.05)  # Huber loss on log latency

# Model Training

In [27]:
from sklearn.metrics import mean_squared_error

In [28]:
def evaluate_mlp(model, loader):
    model.eval()

    preds = []
    targets = []

    with torch.no_grad():
        for model_x, hw_x, y in loader:
            model_x = model_x.to(device)
            hw_x = hw_x.to(device)
            y = y.to(device)

            pred = model(model_x, hw_x)

            preds.append(pred.cpu().numpy())
            targets.append(y.cpu().numpy())

    y_pred = np.concatenate(preds)
    y_true = np.concatenate(targets)

    rmse_log = np.sqrt(mean_squared_error(y_true, y_pred))

    true_ms = np.exp(y_true)
    pred_ms = np.exp(y_pred)

    rel_err = np.abs(pred_ms - true_ms) / true_ms

    metrics = {
        "rmse_log": rmse_log,
        "rmse_ms": np.sqrt(mean_squared_error(true_ms, pred_ms)),
        "rmse_percent": np.sqrt(mean_squared_error(rel_err, np.zeros_like(rel_err))),
        "median_relative_error": np.median(rel_err),
        "p90_relative_error": np.percentile(rel_err, 90),
        "p95_relative_error": np.percentile(rel_err, 95),
        "within_10pct": np.mean(rel_err <= 0.10),
        "median_ratio_error": np.median(np.maximum(pred_ms / true_ms, true_ms / pred_ms)),
        "p90_ratio_error": np.percentile(np.maximum(pred_ms / true_ms, true_ms / pred_ms), 90),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
        "within_2x": np.mean(np.maximum(pred_ms / true_ms, true_ms / pred_ms) <= 2.0),
    }

    return metrics, y_true, y_pred

In [ ]:
best_val_rmse = float("inf")
best_within_10 = -float("inf")
best_state = None

num_epochs = 1600
patience = 300
min_delta = 1e-5
wait = 0

history = []

for epoch in range(1, num_epochs + 1):
    model.train()
    train_losses = []

    for model_x, hw_x, y in train_loader:
        model_x = model_x.to(device)
        hw_x = hw_x.to(device)
        y = y.to(device)

        optimizer.zero_grad()

        pred = model(model_x, hw_x)
        loss = loss_fn(pred, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_losses.append(loss.item())

    scheduler.step(epoch)

    val_metrics, _, _ = evaluate_mlp(model, val_loader)
    val_rmse = val_metrics["rmse_log"]
    val_within_10 = val_metrics["within_10pct"]

    history.append({
        "epoch": epoch,
        "train_loss": np.mean(train_losses),
        "lr": optimizer.param_groups[0]["lr"],
        **val_metrics,
    })

    improved = (
        val_within_10 > best_within_10 + min_delta
        or (abs(val_within_10 - best_within_10) <= min_delta and val_rmse < best_val_rmse - min_delta)
    )

    if improved:
        best_within_10 = val_within_10
        best_val_rmse = val_rmse
        best_state = {
            "model": {k: v.detach().cpu().clone() for k, v in model.state_dict().items()},
            "epoch": epoch,
            "val_rmse": val_rmse,
            "val_within_10pct": val_within_10,
        }
        wait = 0
    else:
        wait += 1

    if epoch % 5 == 0 or epoch == 1:
        print(
            f"epoch {epoch:04d} | "
            f"train_loss={np.mean(train_losses):.5f} | "
            f"val_rmse_log={val_rmse:.5f} | "
            f"within_10={val_within_10:.4f} | "
            f"best_within_10={best_within_10:.4f} | "
            f"lr={optimizer.param_groups[0]['lr']:.2e}"
        )

    if wait >= patience:
        print(f"Early stopping at epoch {epoch}; best epoch {best_state['epoch']}")
        break

# Evaluation

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history)

fig, ax1 = plt.subplots(figsize=(10, 5))

ax1.plot(
    history_df["epoch"],
    history_df["train_loss"],
    label="Train loss",
    color="tab:blue",
)
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Train loss", color="tab:blue")
ax1.tick_params(axis="y", labelcolor="tab:blue")

ax2 = ax1.twinx()
ax2.plot(
    history_df["epoch"],
    history_df["within_10pct"],
    label="Within 10%",
    color="tab:orange",
)
ax2.plot(
    history_df["epoch"],
    history_df["rmse_log"],
    label="Val RMSE log",
    color="tab:green",
    alpha=0.6,
)
ax2.set_ylabel("Validation metrics")
ax2.set_ylim(0, max(1.0, history_df["rmse_log"].max() * 1.1))

fig.suptitle("Training Loss, Within-10%, and RMSE Over Time")
fig.tight_layout()
plt.show()

history_df.sort_values(["within_10pct", "rmse_log"], ascending=[False, True]).head(10)

In [ ]:
model.load_state_dict(best_state["model"])
print("Best epoch:", best_state["epoch"])
print("Best val RMSE:", best_state["val_rmse"])
print("Best val within_10pct:", best_state["val_within_10pct"])

In [ ]:
val_metrics, val_true_log, val_pred_log = evaluate_mlp(model, val_loader)
test_metrics, test_true_log, test_pred_log = evaluate_mlp(model, test_loader)

pd.DataFrame([val_metrics, test_metrics], index=["val", "test"])

# Save Model

In [ ]:
from pathlib import Path
import torch

model_dir = Path("/content/drive/MyDrive/Models")
model_dir.mkdir(parents=True, exist_ok=True)

save_path = model_dir / "13.pt"

if "best_state" in globals() and best_state is not None:
    model.load_state_dict(best_state["model"])

torch.save(
    {
        "model_state_dict": model.state_dict(),
        "n_model_features": X_model_train.shape[1],
        "n_hw_features": X_hw_train.shape[1],
        "model_feature_cols": model_feature_cols,
        "hardware_feature_cols": hardware_feature_cols,
        "model_scaler": model_scaler,
        "hw_scaler": hw_scaler,
        "target_col": "log_average_ms",
        "best_val_rmse": best_val_rmse if "best_val_rmse" in globals() else None,
        "best_val_within_10pct": best_within_10 if "best_within_10" in globals() else None,
        "seed": SEED,
    },
    save_path,
)

print("Saved to:", save_path)

# Analysis

In [29]:
from pathlib import Path
import torch

load_path = Path("/content/drive/MyDrive/Models/13.pt")
try:
    checkpoint = torch.load(load_path, map_location=device, weights_only=False)
except TypeError:
    checkpoint = torch.load(load_path, map_location=device)

model = GatedFiLMLatencyMLP(
    n_model_features=checkpoint["n_model_features"],
    n_hw_features=checkpoint["n_hw_features"],
).to(device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

model_feature_cols = checkpoint["model_feature_cols"]
hardware_feature_cols = checkpoint["hardware_feature_cols"]
model_scaler = checkpoint["model_scaler"]
hw_scaler = checkpoint["hw_scaler"]
target_col = checkpoint["target_col"]

X_model_test = test_df[model_feature_cols].astype("float32").to_numpy()
X_hw_test = test_df[hardware_feature_cols].astype("float32").to_numpy()
y_test = test_df[target_col].astype("float32").to_numpy()

X_model_test = model_scaler.transform(X_model_test).astype("float32")
X_hw_test = hw_scaler.transform(X_hw_test).astype("float32")

test_loader = DataLoader(
    TwoTowerLatencyDataset(X_model_test, X_hw_test, y_test),
    batch_size=batch_size,
    shuffle=False,
    num_workers=0,
)

test_metrics, test_true_log, test_pred_log = evaluate_mlp(model, test_loader)
pd.DataFrame([test_metrics], index=["test"])

print("Loaded model from:", load_path)

Loaded model from: /content/drive/MyDrive/Models/13.pt


In [49]:
test_results = test_df.copy()

test_results["pred_log_latency"] = test_pred_log
test_results["true_log_latency"] = test_true_log

test_results["pred_latency_ms"] = np.exp(test_results["pred_log_latency"])
test_results["true_latency_ms"] = np.exp(test_results["true_log_latency"])

test_results["relative_error"] = (
    np.abs(test_results["pred_latency_ms"] - test_results["true_latency_ms"])
    / test_results["true_latency_ms"]
)

In [31]:
def group_latency_metrics(g):
    y_true = g["true_log_latency"].to_numpy()
    y_pred = g["pred_log_latency"].to_numpy()

    true_ms = np.exp(y_true)
    pred_ms = np.exp(y_pred)

    rel_err = np.abs(pred_ms - true_ms) / true_ms

    return pd.Series({
        "count": len(g),
        "rmse_log": np.sqrt(mean_squared_error(y_true, y_pred)),
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),
    })

In [32]:
test_results["abs_log_error"] = np.abs(
    test_results["pred_log_latency"] - test_results["true_log_latency"]
)

test_results.groupby(["cpu_provider", "platform", "num_cores"])["relative_error"].agg(
    count="count",
    median="median",
    p90=lambda x: x.quantile(0.9),
    within_10pct=lambda x: (x <= 0.10).mean(),
    within_25pct=lambda x: (x <= 0.25).mean(),
).reset_index()

,cpu_provider,platform,num_cores,count,median,p90,within_10pct,within_25pct
0,amd,gcloud,1,2621,0.016447,0.070843,0.919878,0.975582
1,amd,gcloud,2,3763,0.035593,0.169077,0.804411,0.952166
2,amd,gcloud,4,2493,0.041682,0.255642,0.754112,0.897714
3,intel,bluehive,1,1347,0.014369,0.067522,0.915367,0.940609
4,intel,bluehive,2,2686,0.013794,0.052941,0.928146,0.938570
5,intel,bluehive,4,3921,0.015708,0.082144,0.910737,0.932160
6,intel,bluehive,8,2347,0.019205,0.086691,0.908394,0.933106
7,intel,gcloud,6,1350,0.018441,0.089827,0.909630,0.930370


In [33]:
test_results.groupby(["cpu_provider", "platform", "num_cores"])["cv"].agg(
    within_10pct=lambda x: (x <= 0.10).mean(),
    count="count",
    median="median",
    p90=lambda x: x.quantile(0.9),
).reset_index()

,cpu_provider,platform,num_cores,within_10pct,count,median,p90
0,amd,gcloud,1,1.0,2621,0.008731,0.026719
1,amd,gcloud,2,1.0,3763,0.029216,0.072814
2,amd,gcloud,4,1.0,2493,0.035029,0.075006
3,intel,bluehive,1,1.0,1347,0.002264,0.006687
4,intel,bluehive,2,1.0,2686,0.004681,0.017785
5,intel,bluehive,4,1.0,3921,0.004827,0.018930
6,intel,bluehive,8,1.0,2347,0.004055,0.015685
7,intel,gcloud,6,1.0,1350,0.007208,0.016773


In [44]:
test_results.groupby(["num_cores"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,num_cores,count,rmse_log,within_10pct,within_25pct,within_50pct
0,1,3968.0,0.149771,0.918347,0.963710,0.986895
1,2,6449.0,0.174888,0.855947,0.946503,0.980462
2,4,6414.0,0.202184,0.849860,0.918771,0.955098
3,6,1350.0,0.221176,0.909630,0.930370,0.951852
4,8,2347.0,0.256191,0.908394,0.933106,0.948871


In [67]:
test_results.groupby(["platform", "cpu_provider"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,platform,cpu_provider,count,rmse_log,within_10pct,within_25pct,within_50pct
0,bluehive,intel,10301.0,0.222010,0.915348,0.935152,0.959713
1,gcloud,amd,8877.0,0.148125,0.824378,0.943787,0.980737
2,gcloud,intel,1350.0,0.221176,0.909630,0.930370,0.951852


In [68]:
test_results.groupby(["cpu_provider"]).apply(
    group_latency_metrics,
    include_groups=False
).reset_index()

,cpu_provider,count,rmse_log,within_10pct,within_25pct,within_50pct
0,amd,8877.0,0.148125,0.824378,0.943787,0.980737
1,intel,11651.0,0.221914,0.914685,0.934598,0.958802


In [64]:
within_10_misses = test_results[test_results["relative_error"] > 0.10].copy()

print("Outside within-10 count:", len(within_10_misses))
print("Outside within-10 share:", len(within_10_misses) / len(test_results))


def miss_case_summary(g):
    misses = g[g["relative_error"] > 0.10]

    total_count = len(g)
    miss_count = len(misses)
    miss_rate = (miss_count / total_count) * 100 if total_count else np.nan

    if miss_count == 0:
        return pd.Series({
            "total_count": total_count,
            "miss_count": miss_count,
            "miss_rate": miss_rate,
            "miss_rate_percent": miss_rate * 100,
            "rmse_log": np.nan,
            "rmse_percent": np.nan,
            "median_percent_error": np.nan,
            "p90_percent_error": np.nan,
            "p95_percent_error": np.nan,
            "max_percent_error": np.nan,
            "median_ratio_error": np.nan,
            "p90_ratio_error": np.nan,
        })

    y_true = misses["true_log_latency"].to_numpy()
    y_pred = misses["pred_log_latency"].to_numpy()
    rel_err = misses["relative_error"].to_numpy()

    ratio_err = np.maximum(
        misses["pred_latency_ms"].to_numpy() / misses["true_latency_ms"].to_numpy(),
        misses["true_latency_ms"].to_numpy() / misses["pred_latency_ms"].to_numpy(),
    )

    return pd.Series({
        "total_count": total_count,
        "miss_count": miss_count,
        "miss_rate": miss_rate,
        "miss_rate_percent": miss_rate * 100,
        "within_10pct": np.mean(rel_err <= 0.10),
        "within_25pct": np.mean(rel_err <= 0.25),
        "within_50pct": np.mean(rel_err <= 0.50),        
    })




Outside within-10 count: 2553
Outside within-10 share: 0.12436671862821512


In [65]:
test_results.groupby("num_cores").apply(
    miss_case_summary,
    include_groups=False,
).reset_index()

,num_cores,total_count,miss_count,miss_rate,miss_rate_percent,within_10pct,within_25pct,within_50pct
0,1,3968.0,324.0,8.165323,816.532258,0.0,0.555556,0.839506
1,2,6449.0,929.0,14.405334,1440.533416,0.0,0.628633,0.864370
2,4,6414.0,963.0,15.014032,1501.403181,0.0,0.458982,0.700935
3,6,1350.0,122.0,9.037037,903.703704,0.0,0.229508,0.467213
4,8,2347.0,215.0,9.160631,916.063059,0.0,0.269767,0.441860


In [59]:
test_results.groupby("cpu_provider").apply(
    miss_case_summary,
    include_groups=False,
).reset_index()


,cpu_provider,total_count,miss_count,miss_rate,miss_rate_percent,rmse_log,rmse_percent,median_percent_error,p90_percent_error,p95_percent_error,max_percent_error,median_ratio_error,p90_ratio_error
0,amd,8877.0,1559.0,17.562239,1756.223950,0.343679,51.048605,18.169531,52.155685,76.740196,716.437744,1.196427,1.591553
1,intel,11651.0,994.0,8.531457,853.145653,0.754918,119.878342,47.774940,130.191544,245.457962,1222.648682,1.522011,3.252572


In [60]:
test_results.groupby(["platform", "cpu_provider"]).apply(
    miss_case_summary,
    include_groups=False,
).reset_index()

,platform,cpu_provider,total_count,miss_count,miss_rate,miss_rate_percent,rmse_log,rmse_percent,median_percent_error,p90_percent_error,p95_percent_error,max_percent_error,median_ratio_error,p90_ratio_error
0,bluehive,intel,10301.0,872.0,8.465198,846.519755,0.758350,115.115947,46.582191,130.091888,236.889297,1035.558594,1.517431,3.288362
1,gcloud,amd,8877.0,1559.0,17.562239,1756.223950,0.343679,51.048605,18.169531,52.155685,76.740196,716.437744,1.196427,1.591553
2,gcloud,intel,1350.0,122.0,9.037037,903.703704,0.729917,149.565672,54.965984,129.364365,246.464615,1222.648682,1.597907,2.952728
